# API Fetch & Exploration of Rent Control (Logement Encadrement des Loyers) Data
---

**Source:** https://opendata.paris.fr/explore/?disjunctive.theme&disjunctive.publisher&disjunctive.keyword&disjunctive.modified&disjunctive.features&sort=modified

**Endpoint used:** `logement-encadrement-des-loyers`  

Imports and Config

In [1]:
import requests
import pandas as pd
import json
from tqdm.notebook import tqdm

# Configuration: select the years to fetch from the API
YEARS = [2025]

BASE_URL  = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets"
DATASET = "logement-encadrement-des-loyers"

## 2. API Response Check

In [2]:
# Create the endpoint URL using .format()
endpoint = "{}/{}/records".format(BASE_URL, "logement-encadrement-des-loyers")

# Perform a GET request
response = requests.get(endpoint)

# Check the status code
print("Status code:", response.status_code)

# Check the keys of the response
data = response.json()
print(data.keys())


Status code: 200
dict_keys(['total_count', 'results'])


In [3]:
# Retrieve the total number of records available
data = response.json()
total_records = data["total_count"]
print("Total records available:", total_records)

Total records available: 17920


In [4]:
# Filter records by year using the correct API field name.
# Field names were inspected from the API response to ensure valid query syntax.
for col in data["results"][0].keys():
    print(col)

annee
id_zone
id_quartier
nom_quartier
piece
epoque
meuble_txt
ref
max
min
ville
code_grand_quartier
geo_shape
geo_point_2d


In [5]:
# Correct endpoint
endpoint = "{}/{}/records".format(BASE_URL, "logement-encadrement-des-loyers")

# Define parameters (test API maximum limit)
# Note: use correct field name 'annee' from API
params = {
    "limit": 1000,
    "where": "annee=2025"
}

# Send NEW request with these parameters
response = requests.get(endpoint, params=params)

# Check status code (200 = OK)
print(response.status_code)


# Print the full API response
# debug errors and understand the structure of the returned data
#test API maximum limit
print(response.json())

400
{'error_code': 'InvalidRESTParameterError', 'message': 'Invalid value for limit API parameter: 1000 was found but -1 <= limit <= 100 is expected.'}


The error message from the cell above shows that the API limit is 100 records per request

In [6]:
# Build year filter dynamically from configuration
# YEARS are converted into 'refine' filters (annee:YYYY)
year_filters = [f"annee:{year}" for year in YEARS]

params = {
    "limit": 100,
    "refine": year_filters
}

# Send request to the API
response = requests.get(endpoint, params=params)

# Check status code  (200 = OK)
print(response.status_code)

# Inspect response structure
data = response.json()
print(data.keys())

# Check how many records were actually returned
print(len(data["results"]))

200
dict_keys(['total_count', 'results'])
100


## 3. Fetch API Data

In [7]:
year_filters = [f"annee:{year}" for year in YEARS]

# Get total number of records for the selected years
params = {
    "limit": 1,
    "refine": year_filters
}

response = requests.get(endpoint, params=params)
response.raise_for_status()

total_records = response.json()["total_count"]
print("Total records available:", total_records)

# Set maximum allowed number of records per request
limit = 100
all_records = []

# Fetch complete dataset
for offset in tqdm(range(0, total_records, limit), desc="Fetching rent control dataset"):
    params = {
        "limit": limit,
        "offset": offset,
        "refine": year_filters
    }

    # Send request to the API
    response = requests.get(endpoint, params=params)
    response.raise_for_status()

    # Convert API response to JSON
    data = response.json()

    # Add records from the current page to the full list
    all_records.extend(data["results"])

# Check total number of fetched records
print("Total records fetched:", len(all_records))

# Check whether the number of fetched records matches the total count from the API response
print("Match with total count:", len(all_records) == total_records)

Total records available: 2560


Fetching rent control dataset:   0%|          | 0/26 [00:00<?, ?it/s]

Total records fetched: 2560
Match with total count: True


In [8]:
# Print one example value from the 'ville' field
print(all_records[0]["ville"])

# Check all unique values in the 'ville' field across the full dataset
ville_values = sorted({record["ville"] for record in all_records})
print(ville_values)


if ville_values == ["PARIS"]:
    print("PARIS is the only value in the 'ville' field across the full dataset.")
else:
    print("The full dataset contains multiple values in the 'ville' field.")


# Check all unique values in the 'annee' field across the full dataset
# Convert year values to integers before comparing them with the YEARS configuration
year_values = sorted({int(record["annee"]) for record in all_records})
print(year_values)

# Confirm that only the selected years are present
if year_values == sorted(YEARS):
    print("The full dataset only contains records for the selected years.")
else:
    print("The full dataset contains years outside the selected configuration.")



PARIS
['PARIS']
PARIS is the only value in the 'ville' field across the full dataset.
[2025]
The full dataset only contains records for the selected years.


In [9]:
# Convert the list of records to a DataFrame
df_rent_control = pd.DataFrame(all_records)

# Check dataset dimensions and column names
print("Shape:", df_rent_control.shape)
print("Columns:", df_rent_control.columns.tolist())

# Display the first 5 rows
df_rent_control.head(5)

Shape: (2560, 14)
Columns: ['annee', 'id_zone', 'id_quartier', 'nom_quartier', 'piece', 'epoque', 'meuble_txt', 'ref', 'max', 'min', 'ville', 'code_grand_quartier', 'geo_shape', 'geo_point_2d']


,annee,id_zone,id_quartier,nom_quartier,piece,epoque,meuble_txt,ref,max,min,ville,code_grand_quartier,geo_shape,geo_point_2d
0,2025,5,38,Porte-Saint-Denis,2,1946-1970,non meublé,27.4,32.9,19.2,PARIS,7511038,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.352282894951218, 'lat': 48.873617660..."
1,2025,11,77,Belleville,1,1971-1990,non meublé,27.8,33.4,19.5,PARIS,7512077,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3875492398498035, 'lat': 48.87153120..."
2,2025,3,64,Chaillot,4,1971-1990,non meublé,26.7,32.0,18.7,PARIS,7511664,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.2916790427412166, 'lat': 48.86843361..."
3,2025,11,39,Porte-Saint-Martin,1,1946-1970,meublé,30.1,36.1,21.1,PARIS,7511039,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3615036473450113, 'lat': 48.87124465..."
4,2025,6,58,Necker,1,1946-1970,meublé,33.5,40.2,23.5,PARIS,7511558,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3107774536393966, 'lat': 48.84271125..."


## Dataset Exploration

In [10]:
# Overview of data types and non-null counts
df_rent_control.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2560 entries, 0 to 2559
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   annee                2560 non-null   object 
 1   id_zone              2560 non-null   int64  
 2   id_quartier          2560 non-null   int64  
 3   nom_quartier         2560 non-null   object 
 4   piece                2560 non-null   int64  
 5   epoque               2560 non-null   object 
 6   meuble_txt           2560 non-null   object 
 7   ref                  2560 non-null   float64
 8   max                  2560 non-null   float64
 9   min                  2560 non-null   float64
 10  ville                2560 non-null   object 
 11  code_grand_quartier  2560 non-null   int64  
 12  geo_shape            2560 non-null   object 
 13  geo_point_2d         2560 non-null   object 
dtypes: float64(3), int64(4), object(7)
memory usage: 280.1+ KB


In [11]:
# Check missing values
print("missing values:\n", df_rent_control.isna().sum())

print("\n")

# Check for duplicate rows excluding geometry columns to avoid error (unhashable type: 'dict')
print("duplicates:", df_rent_control.drop(columns=["geo_shape", "geo_point_2d"]).duplicated().sum())

missing values:
 annee                  0
id_zone                0
id_quartier            0
nom_quartier           0
piece                  0
epoque                 0
meuble_txt             0
ref                    0
max                    0
min                    0
ville                  0
code_grand_quartier    0
geo_shape              0
geo_point_2d           0
dtype: int64


duplicates: 0


In [12]:
# Summary statistics for rent-related numeric variables
print(df_rent_control[["piece", "ref", "max", "min"]].describe())

print(
    "\n Summary statistics for the rent reference values seem to be plausible.\n "
    "The number of rooms ('piece') ranges from 1 to 4. "
    "Mean values are 27.5 €/m² for ref, 19.3 €/m² for min, and 33 €/m² for max."
)

             piece          ref          max          min
count  2560.000000  2560.000000  2560.000000  2560.000000
mean      2.500000    27.956563    33.546992    19.574414
std       1.118252     4.561769     5.475056     3.192757
min       1.000000    15.100000    18.100000    10.600000
25%       1.750000    24.700000    29.600000    17.300000
50%       2.500000    27.800000    33.400000    19.500000
75%       3.250000    30.800000    37.000000    21.600000
max       4.000000    41.800000    50.200000    29.300000

 Summary statistics for the rent reference values seem to be plausible.
 The number of rooms ('piece') ranges from 1 to 4. Mean values are 27.5 €/m² for ref, 19.3 €/m² for min, and 33 €/m² for max.


#### Translating columns and modalities for ease of analysis.

In [13]:
# Translate columns from French-English for ease of analysis.

# French-English Dictionary
rent_control_dictionary = {
    "annee": "year",
    "id_zone": "zone_id",
    "id_quartier": "quarter_id",
    "nom_quartier": "quarter_name",
    "piece": "room_count",
    "epoque": "construction_period",
    "meuble_txt": "furnished_unfurnished",
    "ref": "reference_rent",
    "max": "max_rent",
    "min": "min_rent",
    "ville": "city",
    "code_grand_quartier": "larger_quarter_code",
    "geo_shape": "geo_shape",
    "geo_point_2d": "geo_point"}

rent_control_translations = pd.DataFrame(
    list(rent_control_dictionary.items()),
    columns=["french_name", "english_translation"]
)

# Check translations
display(rent_control_translations)

# Rename columns
df_rent_control = df_rent_control.rename(columns=rent_control_dictionary)


,french_name,english_translation
0,annee,year
1,id_zone,zone_id
2,id_quartier,quarter_id
3,nom_quartier,quarter_name
4,piece,room_count
5,epoque,construction_period
6,meuble_txt,furnished_unfurnished
7,ref,reference_rent
8,max,max_rent
9,min,min_rent


In [14]:
# Replace modalities in `furnished_unfurnished` with English translations

df_rent_control['furnished_unfurnished'] = df_rent_control['furnished_unfurnished'].replace({
    'meublé': 'furnished',
    'non meublé': 'unfurnished'})

#### Column modifications

In [15]:
# Drop `city` as it has only one unique value, "PARIS"
df_rent_control = df_rent_control.drop(columns=["city"])

if "city" not in df_rent_control.columns:
    print("Column 'city' has been dropped successfully.")
else:
    print("Column 'city' still exists in the DataFrame.")

Column 'city' has been dropped successfully.


In [16]:
# Confirm uniform length of `larger_quarter_code`
for value in df_rent_control['larger_quarter_code']:
  if len(str(value)) != 7:
    print("One or more values in 'larger_quarter_code' have a length other than 7.")
    break
else:
  print("All values in 'larger_quarter_code' have a length of 7.")

All values in 'larger_quarter_code' have a length of 7.


In [17]:
# Split `larger_quarter_code` into postal code and quarter code, then verify that the split value matches with quarter_id before dropping
df_rent_control['postal_code'] = df_rent_control['larger_quarter_code'].astype(str).str[:5]
df_rent_control['quarter_code'] = df_rent_control['larger_quarter_code'].astype(str).str[5:]


In [18]:
# Convert value back to integer
df_rent_control['quarter_code'] = df_rent_control['quarter_code'].astype(int)

# Verify accuracy of quarter code against quarter_id
for index, row in df_rent_control.iterrows():
  if row['quarter_code'] != (row['quarter_id']):
    print("One or more value pairs do not match.")
    break
else:
  print("All value pairs match.")

All value pairs match.


In [19]:
# Drop `quarter_code` column
df_rent_control = df_rent_control.drop(columns=["quarter_code"])

In [20]:
# Drop `larger_quarter_code`
df_rent_control = df_rent_control.drop(columns=["larger_quarter_code"])

# Check if dropped
if "larger_quarter_code" not in df_rent_control.columns:
    print("Column 'larger_quarter_code' has been dropped successfully.")
else:
    print("Column 'larger_quarter_code' still exists in the DataFrame.")

Column 'larger_quarter_code' has been dropped successfully.


#### Filtering values according to the needs of the analysis.


*   `furnished_unfurnished`: unfurnished values only
*   `construction_period`: do a grouping and take the average of all periods.



In [21]:
# Filter out furnished values
df_rent_control = df_rent_control[df_rent_control['furnished_unfurnished'] == 'unfurnished']

# Verify filter success
df_rent_control['furnished_unfurnished'].value_counts()

,count
furnished_unfurnished,
unfurnished,1280


Grouping of data to have values for all construction periods.


*   **min_rent** = minimum of all values for `room_count`, `quarter_id`
*   **reference_rent** = mean of all values for `room_count`, `quarter_id`
*   **max_rent** = max of all values for `room_count`, `quarter_id`

This should leave us with 320 values
( 320 / 80 quarters / 4 room categories ).

In [22]:
# Group to take mean of reference_rent, minimum of min_rent and max of max_rent
df_rent_control_grouped = df_rent_control.groupby(['year', 'zone_id', 'postal_code', 'quarter_id', 'quarter_name',
                                           'room_count']).agg({'min_rent':'min',
                                                                                            'reference_rent':'mean',
                                                                                            'max_rent':'max'})
display(df_rent_control_grouped)

min_rent  \
year zone_id postal_code quarter_id quarter_name          room_count             
2025 1       75106       23         Notre-Dame-des-Champs 1               23.5   
                                                          2               20.2   
                                                          3               20.0   
                                                          4               18.9   
             75107       25         Saint-Thomas-d'Aquin  1               23.5   
...                                                                        ...   
     14      75119       76         Combat                4               12.6   
             75120       79         Père-Lachaise         1               17.0   
                                                          2               15.5   
                                                          3               14.4   
                                                          4               12.6   

                                                                      reference_rent  \
year zone_id postal_code quarter_id quarter_name          room_count                   
2025 1       75106       23         Notre-Dame-des-Champs 1                   34.900   
                                                          2                   30.425   
                                                          3                   29.125   
                                                          4                   29.050   
             75107       25         Saint-Thomas-d'Aquin  1                   34.900   
...                                                                              ...   
     14      75119       76         Combat                4                   19.975   
             75120       79         Père-Lachaise         1                   26.125   
                                                          2                   23.425   
                                                          3                   21.350   
                                                          4                   19.975   

                                                                      max_rent  
year zone_id postal_code quarter_id quarter_name          room_count            
2025 1       75106       23         Notre-Dame-des-Champs 1               44.0  
                                                          2               37.6  
                                                          3               36.2  
                                                          4               37.8  
             75107       25         Saint-Thomas-d'Aquin  1               44.0  
...                                                                        ...  
     14      75119       76         Combat                4               26.3  
             75120       79         Père-Lachaise         1               32.8  
                                                          2               29.5  
                                                          3               26.8  
                                                          4               26.3  

[320 rows x 3 columns]

In [23]:
import json

# Select unique geometry columns and their quarter identifiers from the original df_rent_control.
# The 'year' and 'room_count' columns are not included here because geo_shape and geo_point
# are static geographical properties of a quarter, independent of year or room count.

# Convert 'geo_shape' and 'geo_point' to strings for hashing in drop_duplicates
temp_df_for_geo = df_rent_control[['zone_id', 'postal_code', 'quarter_id', 'quarter_name', 'geo_shape', 'geo_point']].copy()
temp_df_for_geo['geo_shape'] = temp_df_for_geo['geo_shape'].apply(json.dumps)
temp_df_for_geo['geo_point'] = temp_df_for_geo['geo_point'].apply(json.dumps)
geo_df = temp_df_for_geo.drop_duplicates()

# Reset the index of df_rent_control_grouped to make the grouping keys accessible as columns for merging
df_rent_control_grouped_reset = df_rent_control_grouped.reset_index()

# Merge the grouped DataFrame with the unique geometry information.
# A left merge is used to ensure all rows from df_rent_control_grouped are kept.
# The merge is performed on the quarter-identifying columns.
df_rent_control_final = df_rent_control_grouped_reset.merge(geo_df,
                                                    on=['zone_id', 'postal_code', 'quarter_id',
                                                        'quarter_name'],
                                                    how='left')

# Display the head and shape of the new DataFrame to confirm the merge was successful
print("Shape of final DataFrame:", df_rent_control_final.shape)
display(df_rent_control_final.head())

Shape of final DataFrame: (320, 11)


,year,zone_id,postal_code,quarter_id,quarter_name,room_count,min_rent,reference_rent,max_rent,geo_shape,geo_point
0,2025,1,75106,23,Notre-Dame-des-Champs,1,23.5,34.900,44.0,"{""type"": ""Feature"", ""geometry"": {""coordinates""...","{""lon"": 2.327356878234607, ""lat"": 48.846427594..."
1,2025,1,75106,23,Notre-Dame-des-Champs,2,20.2,30.425,37.6,"{""type"": ""Feature"", ""geometry"": {""coordinates""...","{""lon"": 2.327356878234607, ""lat"": 48.846427594..."
2,2025,1,75106,23,Notre-Dame-des-Champs,3,20.0,29.125,36.2,"{""type"": ""Feature"", ""geometry"": {""coordinates""...","{""lon"": 2.327356878234607, ""lat"": 48.846427594..."
3,2025,1,75106,23,Notre-Dame-des-Champs,4,18.9,29.050,37.8,"{""type"": ""Feature"", ""geometry"": {""coordinates""...","{""lon"": 2.327356878234607, ""lat"": 48.846427594..."
4,2025,1,75107,25,Saint-Thomas-d'Aquin,1,23.5,34.900,44.0,"{""type"": ""Feature"", ""geometry"": {""coordinates""...","{""lon"": 2.3255876525808024, ""lat"": 48.85526326..."


#### Exploring how to map the data to our DVF set.

In [24]:
# Explore modalities for geo_shape
geo_shape_values = df_rent_control['geo_shape'].value_counts()

print(f"There are {len(geo_shape_values)} unique geo_shape values in the dataset.")


There are 80 unique geo_shape values in the dataset.


In [30]:
# Check if geographic_shape unique to quarters
num_quarters = df_rent_control['quarter_name'].nunique()
print(f"There are {num_quarters} unique quarter values in the dataset.\n")

# Convert the 'geographic_shape' dictionaries to strings before counting unique values
quarters_with_shape = df_rent_control.groupby('quarter_name')['geo_shape'].apply(lambda x: x.apply(json.dumps).nunique())

# Iterate over values to check
quarters_match_shape = True
for quarter, num_shapes in quarters_with_shape.items():
  if num_shapes > 1:
    print("Quarters are not unique to the geographic shapes.")
    quarters_match_shape = False
    break

if quarters_match_shape:
  print("Each quarter has one unique geographic shape.\n")


# Check if quarters take only one rent control zone.
quarters_zone = df_rent_control.groupby('quarter_name')['zone_id'].nunique()

quarters_zone_unique = True
for quarter, num_zones in quarters_zone.items():
  if num_zones != 1:
    print("Quarters do not take one unique zone value.")
    quarters_zone_unique = False
    break

if quarters_zone_unique:
  print("Each quarter takes one unique zone value. Quarter can be used to map to zone.")

There are 80 unique quarter values in the dataset.

Each quarter has one unique geographic shape.

Each quarter takes one unique zone value. Quarter can be used to map to zone.


Based off of these results, we will assign quarters to the records in the `Property Value` dataset to match to the correct rent control zone.

## Exporting the dataset.

In [33]:
df_rent_control.head()

,year,zone_id,quarter_id,quarter_name,room_count,construction_period,furnished_unfurnished,reference_rent,max_rent,min_rent,geo_shape,geo_point,postal_code
0,2025,5,38,Porte-Saint-Denis,2,1946-1970,unfurnished,27.4,32.9,19.2,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.352282894951218, 'lat': 48.873617660...",75110
1,2025,11,77,Belleville,1,1971-1990,unfurnished,27.8,33.4,19.5,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3875492398498035, 'lat': 48.87153120...",75120
2,2025,3,64,Chaillot,4,1971-1990,unfurnished,26.7,32.0,18.7,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.2916790427412166, 'lat': 48.86843361...",75116
8,2025,10,42,Saint-Ambroise,4,Apres 1990,unfurnished,22.4,26.9,15.7,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.376118055918989, 'lat': 48.862345023...",75111
9,2025,6,65,Ternes,4,1971-1990,unfurnished,24.2,29.0,16.9,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.289963738115156, 'lat': 48.881177550...",75117


In [38]:
# reset the index so as to not export index column
df_rent_control_final = df_rent_control.set_index('quarter_id').sort_index()

# last preview of dataframe before export
display(df_rent_control_final.head())

# shape
df_rent_control_final.shape

,year,zone_id,quarter_name,room_count,construction_period,furnished_unfurnished,reference_rent,max_rent,min_rent,geo_shape,geo_point,postal_code
quarter_id,,,,,,,,,,,,
1,2025,2,St-Germain-l'Auxerrois,1,1971-1990,unfurnished,31.7,38.0,22.2,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3349103292801994, 'lat': 48.86065013...",75101
1,2025,2,St-Germain-l'Auxerrois,3,1971-1990,unfurnished,24.5,29.4,17.2,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3349103292801994, 'lat': 48.86065013...",75101
1,2025,2,St-Germain-l'Auxerrois,1,1946-1970,unfurnished,33.9,40.7,23.7,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3349103292801994, 'lat': 48.86065013...",75101
1,2025,2,St-Germain-l'Auxerrois,3,1946-1970,unfurnished,23.4,28.1,16.4,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3349103292801994, 'lat': 48.86065013...",75101
1,2025,2,St-Germain-l'Auxerrois,4,Avant 1946,unfurnished,27.2,32.6,19.0,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3349103292801994, 'lat': 48.86065013...",75101


(1280, 12)

In [39]:
# Save dataset as CSV
df_rent_control_final.to_csv("../data/api_rent_control_2025.csv")
df_rent_control_final.shape

(1280, 12)

In [41]:
# Export to session files
df_rent_control_final.to_csv("../data/api_rent_control_2025.csv")

In [40]:
import os

output_path = "../data/api_rent_control_2024_2025.csv"

# Create the directory if it doesn't exist
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

df_rent_control.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/Real Estate in Paris/Data/api_rent_control_2024_2025.csv
